# Post-Hoc Evaluation of Trained Model
This notebook performs post-hoc evaluation of a trained model using W&B. It includes validation dataset evaluation and plot generation.

## Setup and Imports

In [51]:
# Import required libraries
# %load_ext autoreload
# %autoreload disable
import h5py
from pathlib import Path
import torch
import lightning as L
import numpy as np
from typing import Type

import wandb
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader
from src.config import pretty_config, find_wandb_run, get_current_config, find_and_download_model, consolidate_base_reeval_configs
from src.evaluation import PlotsCallback
from src.plotters.printing_plots import single_window_lines_plotly
import src.models
import src.data_loaders
from src.models import FlowModule
# Set up logging
import logging
from src.logging_util import handler
import plotly.io as pio
import plotly.express as px

pio.templates.default = "ggplot2"


log_level = logging.DEBUG
logging.getLogger('src').setLevel(logging.DEBUG)
logging.basicConfig(level=log_level, handlers=[handler])
logger = logging.getLogger(__name__)

### Select run and model artifact

In [38]:
FIND_RUN = "seq_normal_smC_BIG4"
TEST_RESULTS_CACHE_NAME = "seq_normal_smC_BIG4_reeval_rk440"
PROJECT = "flowtoy"


selected_run = find_wandb_run(FIND_RUN)
assert isinstance(selected_run, wandb.apis.public.Run), f"Run {FIND_RUN} not found" # type: ignore


DEBUG:src.config:Could not find run by ID (seq_normal_smC_BIG4). Will try by name.
DEBUG:src.config:Found run by name (seq_normal_smC_BIG4)


 ✅ Found Run ID: ww928r4g, Run Name: seq_normal_smC_BIG4
 Created at: 2025-06-07T17:48:19Z
   Run URL: https://wandb.ai/tresoor/flowtoy/runs/ww928r4g


In [39]:
# clean up artifacts
api = wandb.Api()

artifacts = selected_run.logged_artifacts()
for art in artifacts:
    logger.info(art.name)
    if art.type == "model":
        if len(art.aliases) == 0 or not ("best" in art.aliases or "latest" in art.aliases):
            logger.warning("Deleting %s", art.name)
            art.delete()
        else:
            logger.info("Keeping %s \n with aliases %s", art.name, art.aliases)


INFO:__main__:2025-06-07-seq_normal_smC_BIG4:v87
INFO:__main__:Keeping 2025-06-07-seq_normal_smC_BIG4:v87 
 with aliases ['best']
INFO:__main__:2025-06-07-seq_normal_smC_BIG4:v108
INFO:__main__:Keeping 2025-06-07-seq_normal_smC_BIG4:v108 
 with aliases ['latest']
INFO:__main__:run-ww928r4g-history:v0


In [40]:
checkpoint_path = find_and_download_model(selected_run, prefer_alias='latest')

2025-06-07-seq_normal_smC_BIG4:v87
  > Type: model, Version: v87, aliases: ['best'], size: 936_896_783, updated: 2025-06-08T01:26:43Z, description: None


wandb: Downloading large artifact 2025-06-07-seq_normal_smC_BIG4:v108, 893.49MB. 1 files... 


2025-06-07-seq_normal_smC_BIG4:v108
  > Type: model, Version: v108, aliases: ['latest'], size: 936_896_783, updated: 2025-06-08T03:38:09Z, description: None


wandb:   1 of 1 files downloaded.  
Done. 0:0:6.5 (137.5MB/s)


Stored model locally in /Users/milan/Code/fusion/experiments/artifacts/2025-06-07-seq_normal_smC_BIG4:v108


In [41]:
# Load configuration and initialize W&B
# Load the configuration file
config = selected_run.config
config['test_cache_name'] = TEST_RESULTS_CACHE_NAME
config['test_cache_mode'] = "use"
# print(pretty_config(config))
eval_run = wandb.init(
    project=PROJECT,
    name=selected_run.name,
    # id=selected_run.id,
    resume="allow",
    mode="disabled",
    config=config,
)

# Load model configuration
C = get_current_config()
print(pretty_config(C))
assert C == config, "Config from wandb run and config from artifact do not match"


{'Attn': [True, False, False, False],
 'Cols': ['FIR_core', 'PD', 'DML', 'POHM', 'Z_axis'],
 'Crop': 1024,
 'data': {'dir': './data/',
          'cols': {'c': ['NBI-median', 'ECRH-median'], 'x': ['FIR_core', 'PD', 'DML', 'POHM', 'Z_axis'], 'meta': ['ShotNum', 'time'], 'label': 'LHD_label'},
          'file': '2024_05_01-NaNsFiltered.parquet',
          'Class': 'FusionShotDataModule',
          'val_shots': [53623, 57732, 64393, 73631, 57094, 63878, 64686, 77089, 75264, 64678, 77598, 64386],
          'batch_size': 128,
          'seq_length': 256,
          'test_shots': [57013, 76702, 77196, 73935, 61028, 64857, 73368, 60814, 77599, 61237, 77409, 76304, 77595, 65481, 77193, 68697, 69514, 68631, 67112,
                         63306, 64770, 60813, 64365, 77604, 77602],
          'crop_margin': 1024,
          'pre_shuffle': True,
          'sample_rate': 10000,
          'train_shots': [26386, 29511, 30043, 30197, 30225, 30262, 30268, 30290, 30310, 31211, 31554, 31650, 31718, 31807, 3

In [42]:

ModelClass = getattr(src.models, C.model.Class)
assert issubclass(ModelClass, FlowModule), "ModelClass must be a subclass of FlowModule"

model = FlowModule.load_from_checkpoint(checkpoint_path)
# Log model summary
# logger.info("Model loaded. Summary:")
# model.log_summary(C)
# Get the number of steps the model has trained

# load checkpoint
epoch = model.current_epoch
logger.info("Model has trained for %d epochs", epoch)

DEBUG:src.models.unet_conditional:[UNet] Initializing ConditionalUNet: input_channels=5, c_channels=2, apex_hidden_channels=64, time_embedding=sinusoidal+mlp, time_embedding_d=32, positional_encoding=sinusoidal+mlp, positional_encoding_d=32, positional_encoding_c=8, ch_mults=[2, 2, 2, 2], is_attn=[True, False, False, False], mid_attn=True, attn_heads=2, n_blocks=2, activation=GELU, norm_groups=4, conditioning=['x_history', 'c', 'position_sequence'], conditioning_method=sequence
DEBUG:src.models.unet_conditional:[UNet] image_proj: in_channels=16, out_channels=64, kernel_size=3, padding=1, ConvLayer=Conv1d
DEBUG:src.models.unet_conditional:[UNet][DownBlock][res=0][block=0] DownBlock: in_channels=64, out_channels=128, time_emb_channels=256, attn=True, attn_heads=2, act=GELU, norm_groups=4, spatial_dim=1
DEBUG:src.models.unet_conditional:[UNet][DownBlock][ResidualBlock] in_channels=64, out_channels=128, time_channels=256, norm_groups=4, act=GELU
DEBUG:src.models.unet_conditional:[UNet][Dow

## Load Test Dataset
Load the validation dataset using the same DataSetClass and parameters as in `run.py`.

In [43]:
# Load validation dataset

DataSetClass: Type[src.data_loaders.FusionShotDataModule] = getattr(src.data_loaders, C.data.Class)
data_module = DataSetClass(**C.data)

data_module.prepare_data()
data_module.setup()
test_df = data_module.data[data_module.data['ShotNum'].isin(data_module.test_shots)]


INFO:src.data_loaders:Loaded 214 shots from 2024_05_01-NaNsFiltered.parquet
INFO:src.data_loaders:Normalizing columns ['FIR_core', 'PD', 'DML', 'POHM', 'Z_axis', 'NBI-median', 'ECRH-median'] with min FIR_core       4.330701e+16
PD             6.914139e-02
DML           -6.145135e-03
POHM           2.196525e+03
Z_axis        -1.534873e-01
NBI-median     0.000000e+00
ECRH-median    0.000000e+00
dtype: float32 and max FIR_core       1.247798e+20
PD             1.000000e+01
DML            4.895297e-03
POHM           9.589799e+05
Z_axis         3.498712e-01
NBI-median     1.134394e+00
ECRH-median    2.161962e+00
dtype: float32
INFO:src.data_loaders:Column 'time_step      ': min=0.0000      max=26161.0000  mean=8090.8289   std=5279.8659   nans=0         
INFO:src.data_loaders:Column 'ShotNum        ': min=26386.0000  max=77604.0000  mean=63184.8728  std=8322.2462   nans=0         
INFO:src.data_loaders:Column 'IP             ': min=30000.4336  max=449670.2812 mean=188392.4531 std=64416.2188 

In [44]:
test_df.groupby('ShotNum')['time_step'].count().sum()


298153

## Data overview plots histograms

In [ ]:
print(len(set(C.data.train_shots))/260)
print(len(set(C.data.val_shots)) / 260)
print(len(set(C.data.test_shots))/260)
# fig = data_module.data.groupby('ShotNum')['time_step'].count().hist(backend="plotly", bins=40)
# fig.show()


In [ ]:
len(set(C.data.test_shots) | set(C.data.train_shots) | set(C.data.val_shots))

### Shot sets histogram

In [ ]:
# Assign categories to each shot number
import pandas as pd
import numpy as np

# Create a mapping from shot number to category
shot_category = {}
df = data_module.data.copy()
shots_included = set(data_module.data['ShotNum'].unique())
num_train = len(shots_included & set(C.data.train_shots))
num_val = len(shots_included & set(C.data.val_shots))
num_test = len(shots_included & set(C.data.test_shots))
total = num_train + num_val + num_test
print("Total:", total)
print("Percentages: Train: {:.2f}, Validation: {:.2f}, Test: {:.2f}".format(num_train/total, num_val/total,num_test/total))
for shot in C.data.train_shots:
    shot_category[shot] = f'Train ({num_train} shots)'
for shot in C.data.val_shots:
    shot_category[shot] = f'Validation ({num_val} shots)'
for shot in C.data.test_shots:
    shot_category[shot] = f'Testing ({num_test} shots)'

# Add a Category column to the DataFrame
category_series = data_module.data['ShotNum'].map(shot_category)
df['Category'] = category_series
print("Unique shots:", df['ShotNum'].nunique())
# Plot histogram with color by category
fig = px.histogram(
    df,
    x=df.groupby('ShotNum')['time_step'].count().values / 10,
    color=df.groupby('ShotNum').apply(lambda x: shot_category.get(x.name, 'unknown')).values,
    labels={
        'color': 'Subset',
        'x': 'Shot Length (ms)',
        'y': 'Count'
    },
    nbins=10,
    log_y=True,
    barmode="group",
)

# Set x-axis ticks at bin centers
# Increase the number of x-ticks for better granularity

# Get all x values from all histogram traces
all_x = np.concatenate([trace.x for trace in fig.data if hasattr(trace, "x")])
x_min, x_max = all_x.min(), all_x.max()
n_ticks = 12  # Increase this for more ticks
x_ticks = np.linspace(600, 2800, n_ticks, dtype=int)
fig.update_xaxes(tickvals=x_ticks)

# Set y-axis tick labels to show actual values on log scale
y_values = list(range(8)) + [10,15, 20,25,30,40]
fig.update_yaxes(
    tickvals=y_values,
    # ticktext=[str(v) for v in y_values],
    type="log"
)

fig.update_layout(
    margin=dict(l=40, r=10, t=30, b=40),
    width=700,
    height=500,
    font=dict(size=14, family="Serif"),
    legend=dict(
        orientation="h",
        yanchor="bottom",
        y=1.02,
        xanchor="right",
        x=1
    )
)
fig.show()
fig.write_image("output/pdfplots/shot_length_histogram.pdf")

## Single window plot

In [45]:
from pprint import pprint
pprint(dict(C.data.cols))

{'c': ['NBI-median', 'ECRH-median'],
 'label': 'LHD_label',
 'meta': ['ShotNum', 'time'],
 'x': ['FIR_core', 'PD', 'DML', 'POHM', 'Z_axis']}


In [46]:
test_loader = data_module.test_dataloader(batch_size_override=1, shuffle=True)
batch = next(iter(test_loader))
meta, conditioning_input, x = batch
pprint(meta)

{'end': tensor([0.5644], dtype=torch.float64),
 'end_i': tensor([5480]),
 'full_history_start': tensor([0.0164], dtype=torch.float64),
 'history_start': tensor([0.5132], dtype=torch.float64),
 'history_start_i': tensor([4968]),
 'shot_number': tensor([69514], dtype=torch.int32),
 'start': tensor([0.5388], dtype=torch.float64),
 'start_i': tensor([5224])}


In [47]:
trainer = L.Trainer()
trainer.model

Using default `ModelCheckpoint`. Consider installing `litmodels` package to enable `LitModelCheckpoint` for automatic upload to the Lightning model registry.
GPU available: False, used: False
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs


### Convenient window query

In [48]:
# alternative
from src.hdf_cache import TestStepHDFCache
test_set = data_module.test_dataset
# df = data_module.test_dataset.data
cache = TestStepHDFCache(C.test_cache_name, 'a')

INFO:src.hdf_cache:Initialized HDF5 cache at output/test_cache/seq_normal_smC_BIG4_reeval_rk440.h5 in mode 'a'


In [53]:
shot_number = 57013
t = 1.08
viable_idxs = cache.find_cached_idxs(shot_number)
batch = test_set.quick_window(shot_number, t, limit_to_idx=viable_idxs, repeat=5)
batch
meta, conditioning_input, x = batch
# x_gen, surr_labels_gen, surr_labels_target = cache.get(meta['shot_number'], meta['start_i'])
# trainer.predict(model, torch.utils.data.DataLoader(batch))
output = model.evaluate(batch, data_module=data_module, n_steps=40)


INFO:src.data_loaders:Shot 57013 at t=1.08 is at start index 10546
INFO:src.data_loaders:Adjusted to available indeces, we get idx 10544 with time 1.0798
DEBUG:src.models.flow:Evaluating batch shape torch.Size([5, 5, 256])
DEBUG:src.models.flow:Integrating path with 40 steps with method rk4
DEBUG:src.models.flow:Solved with 40 steps.
INFO:src.metrics.evaluate_modes:Generating surrogate labels (batched) for batch of 5 samples with max length 10800
DEBUG:src.metrics.peak_metric:Computing pred and target peak properties for 5 samples satisfying condition 'H_only_Wh'.
DEBUG:src.metrics.peak_metric:Done. Updating state.
DEBUG:src.metrics.peak_metric:State updated.
DEBUG:src.metrics.peak_metric:Computing metrics for 5 hits for condition H_only_Wh
DEBUG:src.metrics.peak_metric:Done 5 hits for condition H_only_Wh
DEBUG:src.metrics.peak_metric:Computing pred and target peak properties for 5 samples satisfying condition 'any_Wh'.
DEBUG:src.metrics.peak_metric:Done. Updating state.
DEBUG:src.metr

In [ ]:

pprint(output.keys())


In [50]:
%reload_ext src.plotters.flow_plots
import src.plotters.flow_plots
import src.metrics.metrics as metrics

In [ ]:
peak_features = {
    "pred_peaks": metrics.batch_get_peakprops(x_gen, 0.1),
    "target_peaks": metrics.batch_get_peakprops(x, 0.1),
}

# Set the default min height for plotly figures
# src.plotters.flow_plots.multi_channel_lines_plotly(meta, x, x_gen, conditioning_input, peak_features=peak_features)
src.plotters.flow_plots.multi_channel_lines_plotly(**output,buttons=True);

In [ ]:
import src.plotters.printing_plots


fig = src.plotters.printing_plots.single_window_lines_plotly(
    target_samples=x, conditioning_input=conditioning_input, labels=conditioning_input.get('label').numpy(), title="Shot {shot_number}"
)

In [ ]:
shot_number = meta['shot_number'].item()
out_file_pdf = f"output/pdfplots/single_window_plot_H_type1big_{shot_number}.pdf"
fig.write_image(out_file_pdf)
print(f"Saved plot for shot {shot_number} to {out_file_pdf}")


In [56]:
from importlib import reload

src.plotters.printing_plots = reload(src.plotters.printing_plots)
fig = src.plotters.printing_plots.multi_sample_single_window_lines_plotly(
    **output, title=f"Shot {shot_number} generated with {selected_run.name}"
)

7


In [55]:
shot_number = meta['shot_number'][0].item()
out_file_pdf = f"output/pdfplots/multisample_H_type1big_{shot_number}-{selected_run.name}.pdf"
fig.write_image(out_file_pdf)
print(f"Saved plot for shot {shot_number} to {out_file_pdf}")

Saved plot for shot 57013 to output/pdfplots/multisample_H_type1big_57013-seq_normal_smC_BIG4.pdf


## Run Evaluation and Generate Plots
Run the evaluation functions on the validation dataset and generate plots using the plot functions defined in `evaluation.py`.

In [ ]:
test_loader = data_module.test_dataloader(batch_size_override=8, shuffle=False)

In [ ]:
batch = next(iter(test_loader))
meta, cond, x = batch
N_STEPS = 3
logger.info("Starting evaluation using %s steps on a batch of %s", N_STEPS, len(x))
evaluation_output = model.evaluate(batch, n_steps=N_STEPS, solve_method='rk4', data_module=data_module)
logger.info("Evaluation completed.")
print(evaluation_output.keys())


In [ ]:
 def get_sequence_info(self, sequence_name):
        if sequence_name in self.file:
            grp = self.file[sequence_name]
            return {
                'available_indices': sorted([int(k) for k in grp.keys()]),
                'count': len(grp.keys()),
                'metadata': dict(grp.attrs)
            }
        return None

## Pipeline for get labels

In [ ]:
from src.evaluate_modes import get_mode_predictions

meta = evaluation_output['meta']
shot_numbers = meta['shot_number']
full_history_start_t = meta['full_history_start']
prediction_window_starts_idx = meta['start_i']
end_time = meta["end"]
PD_index = C.data.cols.x.index("PD")
history_length = C.data.history_length
seq_length = C.data.seq_length
surr_labels_target = []
surr_labels_pred = []
for i, shot_num in enumerate(shot_numbers):
    full_history_raw = val_set.get_full_history(shot_num.item(), prediction_window_starts_idx[i].item())
    full_history = val_set.denormalize(full_history_raw) # type: ignore
    target_samples_raw = val_set.denormalize(evaluation_output['target_samples'][i])
    generated_samples_raw = val_set.denormalize(evaluation_output['generated_samples'][i])
    target_pd_rollout = torch.concat((full_history, target_samples_raw), dim=-1)[PD_index]  # type: ignore
    predicted_pd_rollout = torch.concat((full_history, generated_samples_raw), dim=-1)[PD_index]  # type: ignore
    idx_timeline = np.arange(-prediction_window_starts_idx[i], seq_length)
    surr_labels_target.append(get_mode_predictions(target_pd_rollout, idx_timeline, history_length, seq_length=seq_length).values)
    surr_labels_pred.append(
        get_mode_predictions(predicted_pd_rollout, idx_timeline, history_length, seq_length=seq_length).values
    )

surr_labels_target = np.stack(surr_labels_target)
surr_labels_pred = np.stack(surr_labels_pred)


## Fake genrations stress test on ssurogate model

In [ ]:
meta = evaluation_output['meta']
shot_numbers = meta['shot_number']
full_history_start_t = meta['full_history_start']
prediction_window_starts_idx = meta['start_i']
end_time = meta["end"]
PD_index = C.data.cols.x.index("PD")
history_length = C.data.history_length
seq_length = C.data.seq_length
surr_labels_target = []
surr_labels_pred = []
evaluation_output['generated_samples'] = 0.2 * torch.randn_like(evaluation_output['generated_samples']) + 0.5
for i, shot_num in enumerate(shot_numbers):
    full_history_raw = val_set.get_full_history(shot_num.item(), prediction_window_starts_idx[i].item())
    full_history = val_set.denormalize(full_history_raw)  # type: ignore
    target_samples_raw = val_set.denormalize(evaluation_output['target_samples'][i])
    generated_samples_raw = val_set.denormalize(evaluation_output['generated_samples'][i])
    # fake generated samples. Constant PD of 0.5
    fake_samples = 0.2 * torch.randn_like(generated_samples_raw) + 0.5
    target_pd_rollout = torch.concat((full_history, target_samples_raw), dim=-1)[PD_index]  # type: ignore
    predicted_pd_rollout = torch.concat((full_history, generated_samples_raw), dim=-1)[PD_index]  # type: ignore
    idx_timeline = np.arange(-prediction_window_starts_idx[i], seq_length)
    surr_labels_target.append(
        get_mode_predictions(target_pd_rollout, idx_timeline, history_length, seq_length=seq_length).values
    )
    surr_labels_pred.append(
        get_mode_predictions(predicted_pd_rollout, idx_timeline, history_length, seq_length=seq_length).values
    )

surr_labels_target = np.stack(surr_labels_target)
surr_labels_pred = np.stack(surr_labels_pred)

## Plot labels

In [ ]:
from typing import Optional
import torch
import wandb
import numpy as np

import plotly.graph_objects as go
from plotly import colors as plt_colors
from plotly.subplots import make_subplots as plotly_make_subplots
def multi_channel_lines_plotly(
    meta: dict[str, torch.Tensor],
    target_samples: torch.Tensor,
    generated_samples: torch.Tensor,
    conditioning_input: dict[str, torch.Tensor],
    peak_features: Optional[dict] = None,
    show_c: bool = True,
    surr_labels_target: Optional[torch.Tensor] = None,
    surr_labels_pred: Optional[torch.Tensor] = None,
    n=5,
    title_base="",
    subtitle="",  # Subtitle for the plot
    buttons=False,
    label_bars=True,
    **kwargs  # catch-all for other arguments from evaluate.py
):
    """Create a simple line plot where each channel is a separate color, shots are overlaid, and predictions are show in dotted lines.

    Args:
        meta (dict): Dictionary containing metadata about the samples.
        target_samples (torch.Tensor): Target samples, shape: [num_samples, num_channels, num_timepoints].
        generated_samples (torch.Tensor): Generated samples, shape: [num_samples, num_channels, num_timepoints].
        conditioning_input (dict): Dictionary containing the conditioning input.
        show_c (bool): Whether to show the conditioning channels, if available.
        n (int, optional): Number of traces to visualize. Defaults to 5.
        title_base (str, optional): Base title for the plot. Defaults to "".
        subtitle (str, optional): Subtitle for the plot. Defaults to "".
        buttons (bool, optional): Whether to add buttons to the plot. Defaults to False.

    The legend is grouped by shot.
    Legend format is: 
        Shot 1 - Target:
            Target, Shot 2: Target, Shot 2: Predicted, ..."
    """
    C = get_current_config()
    CHANNEL_NAMES = C.data.cols.x
    history_length = C.data.history_length
    seq_length = C.data.seq_length
    # Generate and visualize new samples
    show_history = "x_history" in conditioning_input
    if show_c and "c" in conditioning_input:
        c_input = conditioning_input["c"] - 1  # Translate everything in c to -1 to 0
        c_channels = c_input.size(1)
        c_axis_values = np.arange(-history_length, seq_length)
        C_CHANNEL_NAMES = C.data.cols.c
    else:
        c_channels = 0  # skips the loop below
        C_CHANNEL_NAMES = []
    position_sequence = conditioning_input["position_sequence"]
    labels = conditioning_input.get('label').numpy()
    shot_numbers = meta["shot_number"]
    start_times = meta["start"]
    end_times = meta["end"]
    num_samples, n_channels, num_timepoints = target_samples.size()
    num_samples = min(n, num_samples)  # Number of traces to visualize
    COLOR_SCALE = plt_colors.qualitative.Plotly
    C_COLOR_SCALE = plt_colors.qualitative.Set2
    mse = ((target_samples[:num_samples] - generated_samples[:num_samples])**2).mean().item()
    subtitle = f"MSE: {mse:.4f}" + (f" | {subtitle}" if subtitle else "")

    # Initialize the figure
    fig = plotly_make_subplots(
        rows=1,
        cols=1,
        specs=[[{
            "secondary_y": True
        }]],
    )
    fig.update_yaxes(
        range=(-1.1, 1.1),
        secondary_y=False,
    )
    fig.update_xaxes(
        range=(-history_length, seq_length), showticklabels=True, title_text="Time steps (0.1ms/step)", showgrid=True
    )
    fig.update_layout(
        title=title_base + (f"<br><sub>{subtitle}</sub>" if subtitle else ""),
        template='plotly_dark',
        hovermode='closest',
        barmode='stack',
        barcornerradius=15,
    )
    if show_history:
        # draw a vertical line around x = 0 to separate conditioning from prediction
        fig.add_shape(
            type="line",
            x0=-0.5,
            x1=-0.5,
            y0=0,
            y1=1,
            line=dict(
                color="yellow",
                width=3,
                dash="solid",
            ),
            xref="x",
            yref="paper",
            opacity=0.5,
        )
        fig.add_shape(
            type="rect",
            x0=-1,
            x1=0,
            y0=0,
            y1=1,
            fillcolor="yellow",
            opacity=0.3,
            line_width=0,
            xref="x",
            yref="paper",
        )
        x_history = conditioning_input['x_history']
    shot_ids = []  # to match buttons to shot number - time identifiers
    for shot_i in range(num_samples):
        start_time = start_times[shot_i]
        shot_sample_id = f"{shot_numbers[shot_i]}:{start_time:.2f}s"
        shot_ids.append(shot_sample_id)
        end_time = end_times[shot_i]
        hover_info_template = "<b>%{y:.5f}</b><br>t: %{x:,}<br><br>" + f"<em>Shot #{shot_sample_id}</em><br>Time span: {start_time:.4f}s-{end_time:.4f}s"
        shot_i_labels = labels[shot_i]

        if label_bars:
            add_mode_bars(fig, history_length, seq_length, num_samples, shot_i_labels, shot_i, shot_sample_id)
            if surr_labels_pred is not None:
                add_mode_bars(
                    fig,
                    history_length,
                    seq_length,
                    num_samples,
                    surr_labels_pred[shot_i],
                    shot_i,
                    shot_sample_id,
                    group='predicted',
                )
            if surr_labels_target is not None:
                add_mode_bars(
                    fig,
                    history_length,
                    seq_length,
                    num_samples,
                    surr_labels_target[shot_i],
                    shot_i,
                    shot_sample_id,
                    group='target',
                )
        # Plot target samples
        for channel_i in range(n_channels):
            channel_color = COLOR_SCALE[((n_channels * shot_i) + channel_i) % len(COLOR_SCALE)]
            channel_name = CHANNEL_NAMES[channel_i]
            target_trace = target_samples[shot_i, channel_i, :].numpy()

            if peak_features:
                pred_peak_features = peak_features['pred_peaks'][shot_i][channel_i]
                target_peak_features = peak_features['target_peaks'][shot_i][channel_i]
                # Find peaks and plot them
                add_peak_markers(
                    fig,
                    target_peak_features,
                    "Target",
                    shot_sample_id,
                    hover_info_template,
                    channel_color,
                    channel_name,
                )
                add_peak_markers(
                    fig,
                    pred_peak_features,
                    "Predicted",
                    shot_sample_id,
                    hover_info_template,
                    channel_color,
                    channel_name,
                )
            # Plot target traces
            fig.add_trace(
                go.Scatter(
                    x=np.arange(seq_length),
                    y=target_trace,
                    mode='lines',
                    line=dict(color=channel_color, width=3),
                    opacity=0.6,
                    customdata=shot_i_labels[history_length:],  # Only show labels for the prediction part
                    name=f'{channel_name} (target)',
                    legendgroup=f'Shot {shot_sample_id} - Target',
                    legendgrouptitle_text=f'Shot {shot_sample_id} - Target',
                    hovertemplate="<b>%{y:.5f}</b><br>t: %{x:,}<br>Label: %{customdata}<br><br>" +
                    f"<em>Shot #{shot_sample_id}</em><br>Time span: {start_time:.4f}s-{end_time:.4f}s"
                )
            )
            # Plot target samples
            if show_history:
                history_start_time = meta['history_start'][shot_i]
                fig.add_trace(
                    go.Scatter(
                        x=np.arange(-history_length, 0),
                        y=x_history[shot_i, channel_i, :],
                        mode='lines',
                        line=dict(color=channel_color, width=3),
                        opacity=0.8,
                        customdata=shot_i_labels[:history_length],  # Only show labels for the history part
                        name=f'{channel_name} (history)',
                        legendgroup=f'Shot {shot_sample_id} - History',
                        legendgrouptitle_text=f'Shot {shot_sample_id} - History',
                        hovertemplate="<b>%{y:.5f}</b><br>t: %{x:,}<br>Label: %{customdata}<br><br>" +
                        f"<em>Shot #{shot_sample_id}</em><br>History time span: {history_start_time:.4f}s-{start_time:.4f}s"
                    )
                )

            # Plot generated traces
            fig.add_trace(
                go.Scatter(
                    x=np.arange(seq_length),
                    y=generated_samples[shot_i, channel_i, :],
                    mode='lines',
                    line=dict(dash='dot', color=channel_color),
                    opacity=0.9,
                    name=f'{channel_name} (predicted)',
                    legendgroup=f'Shot {shot_sample_id} - Predicted',
                    legendgrouptitle_text=f'Shot {shot_sample_id} - Predicted',
                    hovertemplate="<b>%{y:.5f}</b><br>t: %{x:,}<br><br>" +
                    f"<em>Shot #{shot_sample_id}</em><br>Time span: {start_time:.4f}s-{end_time:.4f}s"
                )
            )
        for channel_j in range(c_channels):
            # Plot C (covariate) traces to the bottom of the plot
            channel_color = C_COLOR_SCALE[channel_j]
            channel_name = C_CHANNEL_NAMES[channel_j]
            fig.add_trace(
                go.Scatter(
                    x=c_axis_values,
                    y=c_input[shot_i, channel_j, :],
                    mode='lines',
                    line=dict(color=channel_color, width=4),
                    opacity=0.7,
                    name=f'{channel_name} (C)',
                    legendgroup=f'Shot {shot_sample_id} - C',
                    legendgrouptitle_text=f'Shot {shot_sample_id} - C',
                    hovertemplate="<b>%{y:.5f}</b><br>t: %{x:,}<br><br>" +
                    f"<em>Shot #{shot_sample_id}</em><br>Time span: {start_time:.4f}s-{end_time:.4f}s"
                )
            )
    if buttons:
        # Add a button to focus on every shot individually, and all shots
        not_predicted = [not trace.legendgroup.endswith('Predicted') for trace in fig.data]
        not_target = [not trace.legendgroup.endswith('Target') for trace in fig.data]
        main_button_list = [
            dict(label='All Shots', method='update', args=[{
                'visible': [True] * len(fig.data)
            }]),
            dict(label='Targets', method='update', args=[{
                'visible': not_predicted
            }]),
            dict(label='Predicted', method='update', args=[{
                'visible': not_target
            }])
        ]
        for shot_sample_id in shot_ids:
            main_button_list.append(
                dict(
                    label=shot_sample_id,
                    args=[{
                        "visible": [shot_sample_id in trace.legendgroup for trace in fig.data]
                    }],
                    method="update"
                )
            )
        channel_buttons = [
            dict(label='All', method='update', args=[{
                'visible': [True] * len(fig.data)
            }]),
        ]
        for channel in (CHANNEL_NAMES + C_CHANNEL_NAMES):
            channel_buttons.append(
                dict(
                    args=[{
                        "visible": [trace.name.startswith(channel) for trace in fig.data]
                    }],
                    label=channel,
                    method="update"
                )
            )

        # update traces such that only the first shot is visible initially
        first_shot = shot_numbers[0].item()
        fig.update_traces(visible=False)
        for trace in fig.data:
            if trace.legendgroup.startswith(f'Shot {first_shot}'):
                trace.visible = True
        fig.update_layout(
            updatemenus=[
                dict(
                    active=3,
                    buttons=main_button_list,
                    showactive=True,
                    direction="down",
                    x=1.02,
                    xanchor="left",
                    y=1.02,
                    yanchor="bottom"
                ),
                dict(
                    active=0,
                    buttons=channel_buttons,
                    showactive=True,
                    direction="down",
                    x=1.019,
                    xanchor="right",
                    y=1.02,
                    yanchor="bottom"
                )
            ]
        )

    if wandb.run.disabled:  # type: ignore
        fig.show()

    return fig


def add_peak_markers(
    fig,
    peak_features,
    group: str,
    shot_number,
    hover_info_template,
    channel_color,
    channel_name,
):
    do_plot_energy_delta = peak_features.energy_base_x is not None
    peak_x_markers = []
    peak_y_markers = []
    peak_width_y_markers = []
    peak_width_x_markers = []
    peak_energy_delta_y = []
    peak_energy_delta_x = []
    for peak in peak_features.iter_peaks():
        peak_x_markers.extend([peak.X, peak.X, None])  # None creates a break between lines
        peak_y_markers.extend([peak.bases, peak.Y, None])
        peak_width_y_markers.extend([peak.bases, peak.bases, None])
        peak_width_x_markers.extend([peak.left_ips, peak.right_ips, None])
        if do_plot_energy_delta:
            peak_energy_delta_y.extend([peak.Y, peak.Y - peak.energy_delta, None])
            peak_energy_delta_x.extend([peak.X, peak.energy_base_x, None])

    if do_plot_energy_delta:
        fig.add_trace(  # Peak energy delta markers
            go.Scatter(
                x=peak_energy_delta_x,
                y=peak_energy_delta_y,
                mode='lines+markers',
                line=dict(
                    color=channel_color,
                    dash="solid",
                    width=1,
                ),
                marker=dict(
                                symbol='triangle-up-dot',
                                size=6,
                                angleref='previous',
                    color=channel_color,
                                standoff=3,
                    opacity=0.8,
                            ),
                # marker=dict(
                #     size=8,
                #     symbol="triangle-down",
                # ),
                opacity=0.7,
                name=f'{channel_name} (energy delta)',
                legendgroup=f'Shot {shot_number} - Peaks {group}',
                legendgrouptitle_text=f'Shot {shot_number} - Peaks {group}',
                hovertemplate=f"<b>{group}</b><br>{hover_info_template}"
            )
        )
    fig.add_trace(  # Peak markers traces
                    go.Scatter(
                        x=peak_x_markers,
                        y=peak_y_markers,
                        mode='markers+lines',
                        marker=dict(
                            size=10,
                            color=channel_color,
                            symbol="circle-open-dot",
                            angleref="previous",
                            opacity=0.8,
                        ),
                        opacity=0.9,
                        line=dict(
                            color=channel_color,
                            dash="solid",
                            width=0.8,
                        ),
                        name=f'{channel_name} (peaks)',
                        legendgroup=f'Shot {shot_number} - Peaks {group}',
                        legendgrouptitle_text=f'Shot {shot_number} - Peaks {group}',
                        hovertemplate=f"<b>{group}</b><br>{hover_info_template}"
                    ),
                    secondary_y=False
                )
    fig.add_trace(  # Peak markers traces
                go.Scatter(
                    x=peak_width_x_markers,
                    y=peak_width_y_markers,
                    mode='markers+lines',
                    line=dict(
                        color=channel_color,
                        dash="solid",
                        width=0.8,
                    ),
                    marker=dict(
                        size=5,
                        color=channel_color,
                        symbol="line-ns-open",
                        opacity=0.4,
                    ),
                    opacity=0.9,
                    name=f'{channel_name} (width)',
                    legendgroup=f'Shot {shot_number} - Peaks {group}',
                    legendgrouptitle_text=f'Shot {shot_number} - Peaks {group}',
                        hovertemplate=f"<b>{group}</b><br>{hover_info_template}"

                ),
                )


def add_mode_bars(fig, history_length, seq_length, num_samples, shot_labels, shot_i, shot_number, group: str = 'human'):
    BAR_WIDTH = 25
    MODE_COLORS = ["grey", "lightskyblue", "orange", "red"]
    MODE_NAMES = ["Unknown", "L", "D", "H"]
    GROUP_Y = {'human': 0, 'target': (BAR_WIDTH + num_samples) * 1, 'predicted': (BAR_WIDTH + num_samples) * 2}
    bar_y_placement = GROUP_Y[group] + shot_i
    fig.update_yaxes(
        range=(-BAR_WIDTH / 2, (BAR_WIDTH + num_samples) * 8),
        showticklabels=False,
        secondary_y=True,
        fixedrange=True,
        showgrid=False
    )
    spans = []
    modes = []
    custom_data = []
    colors = []
    current_label = shot_labels[0]
    start_t = -history_length
    for ti in range(0, history_length + seq_length):
        if shot_labels[ti] != current_label:
            next_t = ti - history_length  # Translate to the original time step
            spans.append(next_t - start_t)
            modes.append(MODE_NAMES[int(current_label)])
            colors.append(MODE_COLORS[int(current_label)])
            custom_data.append([shot_number, start_t, next_t, MODE_NAMES[int(current_label)]])
            current_label = shot_labels[ti]
            start_t = next_t
            # Add the last range
    spans.append(seq_length - start_t)
    modes.append(MODE_NAMES[int(current_label)])
    colors.append(MODE_COLORS[int(current_label)])
    custom_data.append([shot_number, start_t, seq_length, MODE_NAMES[int(current_label)]])
    # Add scatter lines for each range
    fig.add_trace(
        go.Bar(
            x=(-history_length, 0),
            y=(bar_y_placement, bar_y_placement),
            orientation='h',
            marker=dict(
                color="black",
                opacity=0,
            ),
            showlegend=False,  # Bar chart does not need a separate legend
            name=f'Shot #{shot_number} - Start',
            legendgroup=f'Shot {shot_number} - Modes',
            hoverinfo='skip',  # Disable hover for this trace
        ),
        secondary_y=True,
    )
    fig.add_trace(
        go.Bar(
            x=spans,
            y=(bar_y_placement,) * len(spans),
            width=BAR_WIDTH,
            orientation='h',
            marker=dict(
                color=colors,
                opacity=0.5 if group=='human' else 0.4,
            ),
            hovertemplate=
            "Mode: %{customdata[3]}<br>Shot #%{customdata[0]}<br>Time steps: %{customdata[1]} - %{customdata[2]}<br>(%{x} steps)",
            customdata=custom_data,
            showlegend=True,  # Bar chart does not need a separate legend
            name=f'Shot #{shot_number} - {group} Labels',
            # hoverinfo=['skip'] + ['all'] * (len(spans) - 1),  # Disable hover for this trace
            legendgroup=f'Shot {shot_number} - Modes',
            legendgrouptitle_text=f'Shot {shot_number} - Modes',
        ),
        secondary_y=True,
    )

In [ ]:
# from src.plotters.flow_plots import multi_channel_lines_plotly
TITLE_POSTFIX =  'newtry'
no_peaks_results = {k: v for k, v in evaluation_output.items() if k != "peak_features"}
# no_peaks_results['generated_samples'] = torch.zeros_like(no_peaks_results['generated_samples']) + 0.5
fig = multi_channel_lines_plotly(
    **no_peaks_results,
    title_base=f"Surrogate label evaluation  {selected_run.name} - {TITLE_POSTFIX}",
    buttons=True,
    surr_labels_pred=surr_labels_pred,
    surr_labels_target=surr_labels_target,
    n=16,
)
Path("output/plots/multiplot").mkdir(parents=True, exist_ok=True)
fig.write_html(f"output/plots/multiplot/slabels-{selected_run.name}- {TITLE_POSTFIX}.html")


# Plot evaluations

In [ ]:
from src.plotters.plot_animations import animated_trajectory_plotly

animated_trajectory_plotly(**evaluation_output, title_base="Animation")

In [ ]:
from plotly import graph_objects as go
import plotly.express as px

def plot_trajectory_plotly(trajectory, meta, title="Trajectory"):
    start_time = meta["full_history_start"].item()
    end_time = meta["end"].item()
    trajectory = trajectory.squeeze(0).cpu().numpy()
    timeline = torch.arange(start_time, end_time, 1e-4).view(1, -1).expand(trajectory.shape[0], -1)
    print(trajectory.shape)
    print(timeline.shape, timeline)
    fig = go.Figure()
    for i, colname in enumerate(C.data.cols.x):
        fig.add_trace(go.Scatter(
            x=timeline[i].cpu().numpy(),
            y=trajectory[i],
            mode='lines+markers',
            name=colname,
            marker=dict(size=5),
        ))
    fig.update_layout(
        title=title,
        xaxis_title='time',
        yaxis_title='value',
        showlegend=True,)
    fig.show()
    

plot_trajectory_plotly(simulated_shot, meta, title="Simulated Shot Trajectory")

In [ ]:
simulated_shot.shape # B, C, T Ex (1, 3, 15000)
# Plotting the results
def plot_trajectory(trajectory, meta, title="Trajectory"):
    start_time = meta["full_history_start"].item()
    end_time = meta["end"].item()
    trajectory = trajectory.squeeze(0).cpu().numpy()
    timeline = torch.arange(start_time, end_time, 1e-4).view(1, -1).expand(trajectory.shape[0], -1)
    print(trajectory.shape)
    print(timeline.shape, timeline)
    plt.figure(figsize=(10, 6))
    plt.plot(timeline.T, trajectory.T, marker='o')
    plt.title(title)
    plt.xlabel('X-axis')
    plt.ylabel('Y-axis')
    plt.grid()
    plt.show()

plot_trajectory(simulated_shot, meta, title="Simulated Shot Trajectory")